# AOS performance
Craig Lage - 18-Jan-26

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time, TimeDelta
from lsst.summit.utils.efdUtils import calcNextDay
from lsst.summit.utils.utils import dayObsIntToString

In [ ]:
dayObss = [20260117]
imageTypes = ['acq', 'science', 'cwfs', 'engtest']

for dayObs in dayObss:
    aosTable = pd.read_json(f'/project/rubintv/LSSTCamAos/sidecar_metadata/shards/dayObs_{dayObs}.json').T
    aosTable = aosTable.sort_index()
    imageTable = pd.read_json(f'/project/rubintv/LSSTCam/sidecar_metadata/dayObs_{dayObs}.json').T
    imageTable = imageTable.sort_index()
    mergedTable = pd.merge(imageTable, aosTable, left_index=True, right_index=True, how='inner')
    mergedTable = mergedTable[mergedTable['Image type_x'].isin(imageTypes)]
mergedTable.index.max()

In [ ]:
for col in mergedTable.columns:
    print(col)

In [ ]:
type(fwhm)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,5))
fwhm = mergedTable['PSF FWHM (median)'].values
clean_fwhm = []
for f in fwhm:
    if not f:
        continue
    if np.isnan(f):
        continue
    clean_fwhm.append(f)
ax.hist(clean_fwhm, bins=20, histtype='step', lw=2)
ax.grid(alpha=0.2)
ax.text(1.2, 100, f'Median = {np.nanmedian(clean_fwhm):.2f}"')
fig.savefig(f"/home/cslage/MTAOS/times_square_reports/FWHM_{dayObs}.png", 
            bbox_inches='tight', pad_inches=1.2)

In [ ]:
aosTable['CalcZernikes config'].tail(20)

In [ ]:
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId, makeDefaultButler
import lsst.summit.utils.butlerUtils as butlerUtils

In [ ]:
butler = butlerUtils.makeDefaultButler("LSSTCam")
client = makeEfdClient()

In [ ]:
seqs = []
states = []
for seq in mergedTable.index:
    expId = int(dayObss[0] * 1E5 + seq)
    print(seq)
    dataId = {'exposure':expId, 'instrument':'LSSTCam'}
    expRecord = getExpRecordFromDataId(butler, dataId)
    rec_start = expRecord.timespan.begin.utc
    event = getMostRecentRowWithDataBefore(
        client,
        "lsst.sal.MTAOS.logevent_degreeOfFreedom",
        timeToLookBefore=Time(rec_start, scale="utc"),
    )
    out = np.empty(
        50,
    )
    for i in range(50):
        out[i] = event[f"aggregatedDoF{i}"]
    states_val = out

    states.append(states_val)
    seqs.append(seq)

# Calculate FWHM Zernike contributions
efd_table = pd.DataFrame(
    {
        "seq": seqs,
        "dof_state": states,
    }
)


In [ ]:
groups = [[0], [5], [1, 2], [6, 7], [3, 4], [8,9]]
group_labels = [' M2 dz', 'Cam dz', 'M2 dx/dy', 'Cam dx/dy', 'M2 tilts', 'Cam tilts']
labels = ['m2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
          'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry']


In [ ]:
dof_state = np.vstack(efd_table["dof_state"].values)
seqs = np.vstack(efd_table["seq"].values)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(31, 18))

linewidth = 0.7
# Top 5x4 GridSpec occupies the upper half
gs_top = gridspec.GridSpec(
    nrows=6, ncols=4,
    width_ratios=[4, 2, 2.15, 2],
    height_ratios=[1]*6,
    hspace=0.0,
    wspace=0.26,
    top=0.97,
    bottom=0.54  # ends halfway down
)

# Bottom 5x4 GridSpec occupies the lower half
gs_bot = gridspec.GridSpec(
    nrows=6, ncols=4,
    width_ratios=[4, 2, 2.15, 2],
    height_ratios=[1]*6,
    hspace=0.0,
    wspace=0.26,
    top=0.46,
    bottom=0.05
)


In [ ]:
axes = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
for id_group, (ax, dof_group) in enumerate(zip(axes, groups)):
    for i in dof_group:
        vals = dof_state[:, i]
        ax.scatter(seqs, vals, label=f"{labels[i]}", s=3)
    ax.set_ylabel(group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")
    leg2 = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
    leg2.get_frame().set_linewidth(0)
    leg2.get_frame().set_edgecolor("none")
    leg2.get_frame().set_facecolor("none")
axes[0].set_title(f'Hexapods State (Trim only)')
axes[-1].set_xlabel("Sequence Number")
fig.savefig(f"/home/cslage/MTAOS/times_square_reports/Trims_{dayObs}.png", 
            bbox_inches='tight', pad_inches=1.2)